# 02 — PhoBERT Sentiment Pipeline

This notebook is a thin execution and inspection layer over the `src/sentiment/` modules.
It does not own preprocessing, MLM training, classifier training, inference, or validation logic.

When running on Kaggle, the notebook can:
- read `GITHUB_TOKEN`, `GEMINI_API_KEY`, and `GDRIVE_SERVICE_ACCOUNT_JSON` from Kaggle Secrets
- clone or pull the real GitHub repo into `/kaggle/working`
- materialize `.dvc/credentials/key.json` and a repo-local `.dvc/config.local` for service-account auth
- optionally `dvc pull` the tracked ViFiC fine-tune source data
- install this repo's `requirements.txt`
- run the existing `src/sentiment/*` modules against that pulled checkout

| Stage | Module |
|---|---|
| Inspect ViFiC schema | `src.sentiment.inspect_vific` |
| Prepare ViFiC and CafeF inputs | `src.sentiment.prepare_inputs` |
| Sample ViFiC annotation subset | `src.sentiment.sample_vific` |
| Annotate with Gemini | `src.sentiment.annotate_vific` |
| Build silver labels | `src.sentiment.build_silver_labels` |
| Optional MLM adaptation | `src.sentiment.pretrain_mlm` |
| Train supervised classifier | `src.sentiment.train_classifier` |
| Infer CafeF sentiment | `src.sentiment.infer_cafef` |
| Validate inference outputs | `src.sentiment.validate_inference` |
| Validate daily aggregation | `src.sentiment.validate_daily_aggregation` |


---
## 0 — Bootstrap

In [ ]:
import base64
import importlib
import json
import os
import subprocess
import sys
from pathlib import Path

import pandas as pd

STAGED_ROOT = Path('..').resolve()
IS_KAGGLE = Path('/kaggle').exists()

def run_command(*args: str, cwd: str | Path | None = None, display: str | None = None) -> None:
    command = [str(arg) for arg in args]
    print('$', display or ' '.join(command))
    subprocess.run(command, check=True, cwd=cwd)

def load_json(path: str | Path) -> dict:
    return json.loads(Path(path).read_text(encoding='utf-8'))

def first_matching_csv(root: str | Path) -> Path | None:
    root_path = Path(root)
    matches = sorted(root_path.rglob('*.csv'))
    return matches[0] if matches else None

def load_kaggle_secret(name: str) -> str | None:
    if not IS_KAGGLE:
        return os.getenv(name)
    try:
        from kaggle_secrets import UserSecretsClient
    except Exception:
        return os.getenv(name)
    try:
        value = UserSecretsClient().get_secret(name)
    except Exception:
        return os.getenv(name)
    return value or os.getenv(name)

def parse_json_secret(value: str | None) -> dict | None:
    if not value:
        return None
    candidates = [value]
    try:
        decoded = base64.b64decode(value).decode('utf-8')
    except Exception:
        decoded = None
    if decoded:
        candidates.append(decoded)
    for candidate in candidates:
        try:
            parsed = json.loads(candidate)
        except Exception:
            continue
        if isinstance(parsed, dict):
            return parsed
    return None

def write_service_account_key(secret_name: str, repo_root: Path) -> Path | None:
    secret_payload = parse_json_secret(load_kaggle_secret(secret_name))
    if not secret_payload:
        return None
    key_path = repo_root / '.dvc' / 'credentials' / 'key.json'
    key_path.parent.mkdir(parents=True, exist_ok=True)
    key_path.write_text(json.dumps(secret_payload, ensure_ascii=False, indent=2), encoding='utf-8')
    return key_path

def configure_dvc_service_account(repo_root: Path, key_path: Path, remote_name: str = 'gdrive') -> Path:
    config_local_path = repo_root / '.dvc' / 'config.local'
    rel_key_path = key_path.relative_to(repo_root / '.dvc')
    config_local_path.write_text(
        f"['remote \"{remote_name}\"']\\n"
        '    gdrive_use_service_account = true\\n'
        f'    gdrive_service_account_json_file_path = {rel_key_path.as_posix()}\\n',
        encoding='utf-8',
    )
    return config_local_path

def github_auth_url(repo_url: str, token: str | None) -> str:
    if not token or not repo_url.startswith('https://github.com/'):
        return repo_url
    return repo_url.replace('https://', f'https://{token}@', 1)
ROOT = STAGED_ROOT

print('Running on Kaggle :', IS_KAGGLE)
print('Staged root       :', STAGED_ROOT)

---
## 1 — Runtime Configuration

In [ ]:
REPO_GIT_URL = 'https://github.com/tlong-ds/news-sentiment-analysis.git'
REPO_BRANCH = 'main'
REPO_DIR_NAME = 'news-sentiment-analysis'
PULL_REPO_ON_KAGGLE = IS_KAGGLE
INSTALL_PROJECT_REQUIREMENTS = IS_KAGGLE
LOAD_KAGGLE_SECRETS = IS_KAGGLE
GDRIVE_SERVICE_ACCOUNT_SECRET = 'GDRIVE_SERVICE_ACCOUNT_JSON'
SETUP_DVC_SERVICE_ACCOUNT = IS_KAGGLE
DVC_PULL_ON_KAGGLE = IS_KAGGLE
DVC_PULL_REMOTE = 'gdrive'
DVC_PULL_TARGETS = ['data/vific/fine-tunes.dvc']

# Set this if the ViFiC source lives outside the repo checkout, for example under /kaggle/input/...
VIFIC_INPUT = None

# Input preparation
MAX_BODY_CHARS = 300
MIN_TOKENS = 5
MAX_TOKENS = 220
VIFIC_PRESEGMENTED = True

# Annotation sampling
SAMPLE_SIZE = 5000
SAMPLE_SEED = 42

# Optional execution gates for expensive or external steps
RUN_ANNOTATION = False
ANNOTATION_DRY_RUN = True
RUN_MLM = False
RUN_CLASSIFIER_TRAINING = False
RUN_CAFEF_INFERENCE = False

# Training/inference hyperparameters
MLM_EPOCHS = 3
MLM_BATCH_SIZE = 32
MLM_LEARNING_RATE = 1e-5
CLASSIFIER_EPOCHS = 10
CLASSIFIER_BATCH_SIZE = 16
CLASSIFIER_LEARNING_RATE = 2e-5
INFERENCE_BATCH_SIZE = 32

---
## 2 — Kaggle Secrets, Repo Pull, And DVC Bootstrap

In [ ]:
if LOAD_KAGGLE_SECRETS:
    for secret_name in ['GITHUB_TOKEN', 'GEMINI_API_KEY']:
        secret_value = load_kaggle_secret(secret_name)
        if secret_value and secret_name not in os.environ:
            os.environ[secret_name] = secret_value

if IS_KAGGLE and PULL_REPO_ON_KAGGLE:
    kaggle_repo_root = Path('/kaggle/working') / REPO_DIR_NAME
    auth_repo_url = github_auth_url(REPO_GIT_URL, os.getenv('GITHUB_TOKEN'))

    if kaggle_repo_root.exists() and (kaggle_repo_root / '.git').exists():
        run_command('git', 'fetch', 'origin', REPO_BRANCH, cwd=kaggle_repo_root, display=f'git fetch origin {REPO_BRANCH}')
        run_command('git', 'checkout', REPO_BRANCH, cwd=kaggle_repo_root)
        run_command('git', 'pull', 'origin', REPO_BRANCH, cwd=kaggle_repo_root, display=f'git pull origin {REPO_BRANCH}')
    else:
        run_command(
            'git', 'clone', '--branch', REPO_BRANCH, auth_repo_url, str(kaggle_repo_root),
            cwd='/kaggle/working',
            display=f'git clone --branch {REPO_BRANCH} {REPO_GIT_URL} {kaggle_repo_root}'
        )
    ROOT = kaggle_repo_root
else:
    ROOT = STAGED_ROOT

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.chdir(ROOT)
importlib.invalidate_caches()

if INSTALL_PROJECT_REQUIREMENTS:
    run_command(sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt', cwd=ROOT)
else:
    print('INSTALL_PROJECT_REQUIREMENTS is False; skipping pip install.')

service_account_key_path = None
dvc_config_local_path = None
if SETUP_DVC_SERVICE_ACCOUNT:
    service_account_key_path = write_service_account_key(GDRIVE_SERVICE_ACCOUNT_SECRET, ROOT)
    if service_account_key_path is None:
        print(f'{GDRIVE_SERVICE_ACCOUNT_SECRET} not found or invalid; skipping DVC service-account setup.')
    else:
        dvc_config_local_path = configure_dvc_service_account(ROOT, service_account_key_path, remote_name=DVC_PULL_REMOTE)
        os.environ.setdefault('DVC_SITE_CACHE_DIR', str(ROOT / '.dvc' / 'site-cache'))
        if DVC_PULL_ON_KAGGLE:
            run_command('dvc', 'pull', '-r', DVC_PULL_REMOTE, *DVC_PULL_TARGETS, cwd=ROOT)
else:
    print('SETUP_DVC_SERVICE_ACCOUNT is False; skipping DVC auth bootstrap.')

from src.config import ANNOTATION_DATA_DIR, CAFEF_DATA_DIR, FINETUNES_DATA_DIR, MODELS_DATA_DIR, PROCESSED_DATA_DIR, VIFIC_NORMALIZED_DIR

def run_module(module: str, *args: str) -> None:
    run_command(sys.executable, '-m', module, *args, cwd=ROOT)

print('ROOT                :', ROOT)
print('GITHUB_TOKEN loaded :', bool(os.getenv('GITHUB_TOKEN')))
print('GEMINI_API_KEY set  :', bool(os.getenv('GEMINI_API_KEY')))
print('DVC SA key written  :', bool(service_account_key_path))
print('DVC config.local    :', dvc_config_local_path)
print('DVC pull targets    :', DVC_PULL_TARGETS if DVC_PULL_ON_KAGGLE else [])
print('ViFiC fine-tunes dir:', FINETUNES_DATA_DIR)
print('ViFiC processed dir :', VIFIC_NORMALIZED_DIR)
print('Annotation dir      :', ANNOTATION_DATA_DIR)
print('CafeF dir           :', CAFEF_DATA_DIR)
print('Processed dir       :', PROCESSED_DATA_DIR)
print('Models dir          :', MODELS_DATA_DIR)

In [ ]:
if VIFIC_INPUT is None:
    discovered_vific = first_matching_csv(FINETUNES_DATA_DIR)
    print('auto-discovered VIFIC_INPUT:', discovered_vific)
else:
    discovered_vific = Path(VIFIC_INPUT)
    print('configured VIFIC_INPUT      :', discovered_vific)

---
## 3 — Inspect ViFiC Source

In [ ]:
inspect_args = []
if VIFIC_INPUT is not None:
    inspect_args.extend(['--input-file', str(VIFIC_INPUT)])

run_module('src.sentiment.inspect_vific', *inspect_args)

schema_summary_path = Path(VIFIC_NORMALIZED_DIR) / 'vific_schema_summary.json'
schema_summary = load_json(schema_summary_path)
schema_summary

---
## 4 — Prepare ViFiC and CafeF Inputs

In [ ]:
prepare_args = [
    '--max-body-chars', str(MAX_BODY_CHARS),
    '--min-tokens', str(MIN_TOKENS),
    '--max-tokens', str(MAX_TOKENS),
]
if VIFIC_INPUT is not None:
    prepare_args.extend(['--vific-input', str(VIFIC_INPUT)])
if not VIFIC_PRESEGMENTED:
    prepare_args.append('--no-vific-presegmented')

run_module('src.sentiment.prepare_inputs', *prepare_args)

input_report_path = Path(VIFIC_NORMALIZED_DIR) / 'input_preparation_report.json'
input_report = load_json(input_report_path)
input_report

In [ ]:
vific_input_path = Path(VIFIC_NORMALIZED_DIR) / 'vific_input.parquet'
vific_pretraining_path = Path(VIFIC_NORMALIZED_DIR) / 'vific_pretraining.parquet'
cafef_input_path = Path(CAFEF_DATA_DIR) / 'cafef_input.parquet'

vific_input_df = pd.read_parquet(vific_input_path)
vific_pretraining_df = pd.read_parquet(vific_pretraining_path)
cafef_input_df = pd.read_parquet(cafef_input_path)

print('vific_input        :', vific_input_df.shape)
print('vific_pretraining  :', vific_pretraining_df.shape)
print('cafef_input        :', cafef_input_df.shape)

vific_input_df.head(3)

---
## 5 — Sample the ViFiC Annotation Subset

In [ ]:
run_module(
    'src.sentiment.sample_vific',
    '--sample-size', str(SAMPLE_SIZE),
    '--seed', str(SAMPLE_SEED),
)

sample_report_path = Path(ANNOTATION_DATA_DIR) / 'vific_annotation_sample_report.json'
sample_report = load_json(sample_report_path)
sample_report

In [ ]:
sample_path = Path(ANNOTATION_DATA_DIR) / 'vific_annotation_sample.parquet'
sample_df = pd.read_parquet(sample_path)
print('sample rows:', len(sample_df))
sample_df[['article_id', 'category', 'date', 'token_count', 'sample_stratum']].head(5)

---
## 6 — Optional Dual-LLM Annotation

In [ ]:
annotation_args = []
if ANNOTATION_DRY_RUN:
    annotation_args.append('--dry-run')

if RUN_ANNOTATION:
    run_module('src.sentiment.annotate_vific', *annotation_args)
    pilot_report_path = Path(ANNOTATION_DATA_DIR) / 'annotation_pilot_report.json'
    load_json(pilot_report_path)
else:
    print('RUN_ANNOTATION is False; skipping annotate_vific.')
    print('Planned command:', [sys.executable, '-m', 'src.sentiment.annotate_vific', *annotation_args])

---
## 7 — Build Silver Labels

In [ ]:
run_module('src.sentiment.build_silver_labels', '--allow-low-kappa')

silver_metrics_path = Path(ANNOTATION_DATA_DIR) / 'silver_label_metrics.json'
silver_metrics = load_json(silver_metrics_path)
silver_metrics

In [ ]:
labeled_dataset_path = Path(ANNOTATION_DATA_DIR) / 'sentiment_labeled_full.parquet'
labeled_df = pd.read_parquet(labeled_dataset_path)
print('labeled dataset shape:', labeled_df.shape)
print(labeled_df['split'].value_counts().to_dict())
print(labeled_df['label'].value_counts().to_dict())
labeled_df.head(3)

---
## 8 — Optional MLM Adaptation

In [ ]:
mlm_args = [
    '--epochs', str(MLM_EPOCHS),
    '--batch-size', str(MLM_BATCH_SIZE),
    '--learning-rate', str(MLM_LEARNING_RATE),
]

if RUN_MLM:
    run_module('src.sentiment.pretrain_mlm', *mlm_args)
    mlm_report_path = Path(MODELS_DATA_DIR) / 'phobert-vific-adapted' / 'pretraining_report.json'
    load_json(mlm_report_path)
else:
    print('RUN_MLM is False; skipping pretrain_mlm.')
    print('Planned command:', [sys.executable, '-m', 'src.sentiment.pretrain_mlm', *mlm_args])

---
## 9 — Optional Supervised Classifier Training

In [ ]:
classifier_args = [
    '--epochs', str(CLASSIFIER_EPOCHS),
    '--batch-size', str(CLASSIFIER_BATCH_SIZE),
    '--learning-rate', str(CLASSIFIER_LEARNING_RATE),
]

if RUN_CLASSIFIER_TRAINING:
    run_module('src.sentiment.train_classifier', *classifier_args)
    evaluation_path = Path(MODELS_DATA_DIR) / 'phobert-sentiment-cafef' / 'evaluation.json'
    load_json(evaluation_path)
else:
    print('RUN_CLASSIFIER_TRAINING is False; skipping train_classifier.')
    print('Planned command:', [sys.executable, '-m', 'src.sentiment.train_classifier', *classifier_args])

---
## 10 — Optional CafeF Inference

In [ ]:
inference_args = [
    '--batch-size', str(INFERENCE_BATCH_SIZE),
]

if RUN_CAFEF_INFERENCE:
    run_module('src.sentiment.infer_cafef', *inference_args)
else:
    print('RUN_CAFEF_INFERENCE is False; skipping infer_cafef.')
    print('Planned command:', [sys.executable, '-m', 'src.sentiment.infer_cafef', *inference_args])

---
## 11 — Validation and Inspection

In [ ]:
sentiment_output_path = Path(PROCESSED_DATA_DIR) / 'article_sentiment_scores.parquet'

if sentiment_output_path.exists():
    run_module('src.sentiment.validate_inference')
    run_module('src.sentiment.validate_daily_aggregation')

    inference_report = load_json(Path(PROCESSED_DATA_DIR) / 'sentiment_inference_validation.json')
    aggregation_report = load_json(Path(PROCESSED_DATA_DIR) / 'daily_aggregation_validation.json')

    print('ready_for_modeling:', inference_report['ready_for_modeling'])
    print('validation checks  :', inference_report['validation_checks'])
    print('aggregation report :', aggregation_report)

    sentiment_df = pd.read_parquet(sentiment_output_path)
    sentiment_df.head(5)
else:
    print(f'Missing {sentiment_output_path}; run inference first.')

---
## Summary

This notebook delegates all substantive logic to the `src/sentiment/` modules.
On Kaggle, it can bootstrap secrets and pull the real GitHub repo before execution.
